In [1]:
import datetime
import io
import os
import requests
import pandas as pd
import uuid
import json

In [2]:
import sys
import inspect

sys.path.append(os.path.join(os.getcwd(), 'tools\\tools_method'))

import tools_method
from tools_method import *

all_functions = [
    name for name, obj in inspect.getmembers(tools_method)
    if inspect.isfunction(obj) and not name.startswith('_')
]

In [3]:
from logger_config import setup_logging
logger = setup_logging(__name__)

In [4]:
from volcenginesdkarkruntime import Ark

In [5]:
# 模型名称
model_name = 'doubao-seed-1-8-251228'

# 大模型配置参数
LLM_API_KEY = '31d61644-628b-4332-9e5c-1987c99f3427'
LLM_BASE_URL = 'https://ark.cn-beijing.volces.com/api/v3'

In [6]:
# 短期上下文
history_chat = []

In [7]:
def init_llm_client(api_key, base_url):
    """构建llm client
    Args:
        api_key: 申请的apikey
        base_url: 模型的base URL

    Returns:
        client
    """
    client = Ark(api_key=api_key,
    # The base URL for model invocation
    base_url=base_url,
    timeout=1800
    )
    return client

In [8]:
client = init_llm_client(LLM_API_KEY, LLM_BASE_URL)

In [9]:
# tools_desc = [
#     {
#         'type': 'function',
#         'function': {
#             'name': 'get_temperature',
#             'description': '''
#             触发条件：当用户需要查询温度时
#             注意：这些都是天气信息
#             '''
#         }
#     },
#     {
#         'type': 'function',
#         'function': {
#             'name': 'get_moisture',
#             'description': '''
#             触发条件：当用户需要查询湿度时
#             注意：这些都是天气信息
#             '''
#         }
#     }
# ]

# # 保存JSON文件
# try:
#     # 打开文件（w表示写入，encoding指定编码避免中文乱码）
#     with open("./skill/tools_desc.json", "w", encoding="utf-8") as f:
#         # indent=4 让JSON文件格式化显示，更易读；ensure_ascii=False 支持中文显示
#         json.dump(tools_desc, f, indent=4, ensure_ascii=False)
#     print("JSON文件保存成功！")
# except Exception as e:
#     print(f"保存失败：{e}")

In [10]:
# 读取JSON Tool desc文件
try:
    # 打开文件（r表示读取，指定utf-8编码）
    with open("./tools/tools_desc.json", "r", encoding="utf-8") as f:
        # 将JSON字符串解析为Python字典
        tools_desc = json.load(f)
except Exception as e:
    print(f"读取失败：{e}")

# 读取JSON Tool params文件
try:
    # 打开文件（r表示读取，指定utf-8编码）
    with open("./tools/tools_params.json", "r", encoding="utf-8") as f:
        # 将JSON字符串解析为Python字典
        tools_params = json.load(f)['function_params']
except Exception as e:
    print(f"读取失败：{e}")

In [11]:
# 参数校验
for desc in tools_desc:
    if desc['function']['name'] not in list(tools_params.keys()):
        raise Exception(f'{desc['function']['name']}没有关联参数信息！')

# 工具检验
for desc in tools_desc:
    if desc['function']['name'] not in all_functions:
        raise Exception(f'{desc['function']['name']}没有具体的执行函数！')

In [12]:
def call_llm_chat_multi_api(client, history_chat, model_name, tool_call=False, thinking='disabled'):
    """接口调用 多轮对话（模型能够解析工具列表，选取工具并给出参数，并解析执行结果，则具备function callingnengli）
    Args:
        client: llm client
        history_chat: 历史对话
        model_name: 模型名或版本
        tool_call: 是否调用工具
        thinking: 是否开启思考, disabled enabled or auto

    Returns:
        response: llm 生成的内容
    """

    # 按顺序提取user和content
    messages = []

    # 系统提示词
    messages.extend([{'role':'system', 'content':'你是我的一名随身助手，帮我解决日常问题！'}])

    # 当前会话的历史记录
    for item in history_chat:
        messages.extend([{'role':item['role'], 
                          'content':item['content'], 
                          'tool_call_id':item.get('tool_call_id', None),
                          'tool_calls':item.get('tool_calls', None)
                          }])

    try:
        # 调用工具
        if tool_call:
            response = client.chat.completions.create(
                # Replace with Model ID .
                model=model_name,
                messages=messages,
                thinking={
                    "type": thinking
                },
                tools=tools_desc, 
                max_completion_tokens = 32768# 40960,
            )
        # 不调用工具，主要针对解析
        else:
            # print("ss")
            response = client.chat.completions.create(
                # Replace with Model ID .
                model=model_name,
                messages=messages,
                thinking={
                    "type": thinking
                },
                max_completion_tokens = 32768 # 40960,
            )
        return {'success':True, 'response': response}
    except Exception as e:
        logger.error(f'LLM计算错误：{str(e)}')
        return {'success':False, 'response': f'LLM计算错误：{str(e)}'}

In [18]:
history_chat = []

In [28]:
# query = "我想查询云南昆明、丽江、西双版纳的天气"
query = "请帮我写一个简单的sql"

# 手写工具调用？
1 根据上下文，判断意图，选择可能需要的工具。大模型判断 or rag？（一个markdown知识库文档）
2 加载工具
3 

In [29]:
history_chat.extend([{'role':'user', 'content':query}])

stop = False

while True:

    # 调用llm提取回答
    result = call_llm_chat_multi_api(client,history_chat,model_name,tool_call=True,thinking='disabled')
    finish_reason = result['response'].choices[0].finish_reason
    # print(finish_reason)
    message = result['response'].choices[0].message.model_dump()
    print(message)

    # 记录llm message答复
    message = {key: message[key] for key in ['content', 'role', 'tool_calls']}
    history_chat.extend([message])

    # 是否需要调用工具
    if message['tool_calls']: 

        # 遍历工具
        for tool in message['tool_calls']:

            # 工具名称 id以及参数
            cur_tool_name = tool['function']['name']
            cur_tool_id = tool['id']
            cur_tool_params = json.dumps(tools_params[cur_tool_name], indent=2, ensure_ascii=False)

            ###########################################################
            ######################参数解析单独提取#######################
            ###########################################################
            # 参数解析提示词
            params_parse_prompt = f"""
            # 参数解析任务
            请你严格按照指定规则，从上文相关内容中解析出符合【{cur_tool_params}】格式要求的参数。

            ## 核心规则（必须严格遵守）
            1. 参数结构：解析结果必须完全匹配定义的参数结构，包括参数名称、数据类型、层级关系。
            2. 必要参数校验：
            - 标记为'required'的参数为**必要参数**，必须从上下文提取完整有效值；
            - 若任意一个必要参数缺失、值为空或无效，则判定为解析失败；
            3. 相关常识性参数你可以自己填；
            4. 输出格式：无论解析成功/失败，均需返回标准JSON格式（注意：键名使用双引号，值为字符串时也使用双引号），具体格式如下：
            {{
                "success": true/false,  // 解析成功（所有必要参数齐全）为true，否则为false
                "content": "",         // 解析成功时为空字符串；解析失败时，明确说明缺失的必要参数，示例："缺失必要参数：start_time、user_id"
                "params": {{}}           // 解析成功时，填入完整的参数键值对；解析失败时，为空对象
            }}

            ## 输出要求
            1. 仅返回JSON字符串，不添加任何额外解释、说明或备注；
            2. JSON格式必须合法可解析，禁止出现语法错误；
            3. 必要参数缺失时，content字段需清晰列出所有缺失的参数名称，无遗漏。
            """
            # 提取参数解析的请求数据 此处不记录到上下文，临时抽取user和assistant的对话信息
            sel_hist_items = [item for item in history_chat if not item.get('tool_calls', None)]
            sel_hist_items.extend([{'role': 'user', 'content': params_parse_prompt}])

            # 调用LLM完成参数解析
            params_result = call_llm_chat_multi_api(client, sel_hist_items,
                                                    model_name, tool_call=False, thinking='disabled')

            # 解析获得的参数
            params_js = params_result['response'].choices[0].message.model_dump()
            print(params_js)

            ###########################################################
            ###########################################################
            ###########################################################
            # 工具调用
            try:

                params = json.loads(params_js['content'])

                # 若解析结果完整，则调用工具
                if params['success']:
                    sel_params = params['params']

                    # 调用工具
                    tool_res = eval(f"{cur_tool_name}(**{sel_params})")
                    history_chat.extend([{'role': 'tool', 'content': tool_res, 'tool': cur_tool_name, 'tool_call_id': cur_tool_id}])
                else:
                    # 返回工具参数缺失信息
                    history_chat.extend([{'role': 'tool', 'content': params['content'], 'tool': cur_tool_name, 'tool_call_id': cur_tool_id}])
            except Exception as e:
                # 返回工具参数错误信息
                print("ssss")
                history_chat.extend([{'role': 'tool', 'content': str(e), 'tool_call_id': cur_tool_id}])

    else:
        print("结束！")
        break

format='2026-02-13 16:59:16,348 - httpx - _send_single_request - INFO - HTTP Request: POST https://ark.cn-beijing.volces.com/api/v3/chat/completions "HTTP/1.1 200 OK"'


{'content': "当然可以！不过还需要你补充一些信息，我才能写出符合需求的SQL语句哦：\n1. **需求场景**：比如是查询数据、插入数据、更新数据还是删除数据？\n2. **涉及的表名和字段**：例如操作哪个表？需要用到哪些字段（如id、name、temperature等）？\n3. **具体条件**：有没有筛选条件（如WHERE子句）、排序要求（ORDER BY）或者聚合需求（如SUM、COUNT）？\n\n举个例子：如果要查询云南所有城市的温度，可以写成：\n```sql\nSELECT city_name, temperature \nFROM weather \nWHERE province = '云南';\n```\n\n你可以告诉我你的具体需求，我来帮你写对应的SQL~", 'role': 'assistant', 'function_call': None, 'tool_calls': None, 'audio': None, 'reasoning_content': None}
结束！


# RAG

1 传统rag会割裂信息，导致上下文割裂，召回信息不完整
2 尽量每一条信息是一个相对独立的内容，如每一条信息就是一张数据表
3 信息检索到知识提炼  信息-知识的跃迁
4 将非结构化信息转为结构化信息，如用langextract

# SKILL

1 结合langchain来做专业开发，利用deepwiki来做开源项目解读？
2 skill

# 计划阶段

用户想要查询"用户在每一年、在每个门店的消费"，这是一个数据分析需求。我需要先了解可用的数据源和数据结构。

从environment_details中可以看到：

当前有两个数据库可用：builtin_file_database 和 mt_brand_data
有一个RAG服务器：mt_rag（mt数据字典）
为了制定一个有效的计划，我需要：

首先使用RAG工具了解业务背景和数据字典
然后探索数据库中的表结构
理解用户、门店、消费相关的表结构
制定查询计划

注意：计划阶段不会涉及表注册和结果生成，只做rag和数据结构探索

1 查询数据字典来了解业务术语和数据表结构

问题：请提供关于用户、门店、消费相关的数据表信息，包括表名、字段定义和业务含义
数据：rag

2 探索数据库中的数据集
问题：想要搜索数据集 mt_brand_data

3 表结构分析

问题：想要获取表结构 mt_brand_data: mt_members.csv

问题：想要获取表结构 mt_brand_data: mt_stores.csv

问题：想要获取表结构 mt_brand_data: mt_consumption.csv

4 查询计划

数据准备阶段：将三个表注册到Infinity Session中

核心查询：按年份和门店分组，统计每个用户在每个门店的消费总额

结果展示：包含用户信息、门店信息、年份、消费金额等关键指标

具体查询逻辑：

从消费表按member_id、store_id和年份分组
计算每个分组的payment字段总和
关联会员表和门店表获取详细信息
按年份和门店进行排序展示
预期输出字段：

用户ID、用户姓名、会员等级
门店ID、门店名称、所在城市
年份、消费总额

# 执行阶段

列计划
用户要求分析MT数据并统计用户数。我需要先了解MT数据库中有哪些表，然后找到包含用户信息的表进行统计。

根据环境信息，我看到有一个名为"mt_brand_data"的数据库，并且有一个相关的RAG服务器"mt_rag"。我应该先使用RAG工具来了解MT数据的业务背景，然后探索数据库结构。
（遵从你对RAG和Data这两个概念理解的本心：
数据源就是数据
RAG就是业务说明/数据词典/分析套路说明/各类规范等等教大模型更好的完成任务的指引,也包括你过往的对数据使用的案例或者文档。）

步骤：
（1）使用RAG工具了解MT数据背景和用户相关的表
（2）列出mt_brand_data数据库中的表
（3）注册相关的用户表
（4）查看表结构
（5）统计用户数
由于这是多步骤任务，我应该创建一个todo列表来跟踪进度。

1 使用RAG工具了解MT数据背景和用户相关的表
问题：MT数据包含哪些表？哪些表包含用户信息？如何统计用户数量？用户表的主要字段有哪些？
数据：mt_rag

注意：知识库：对表信息的介绍文件，包含字段说明、小样本等信息。知识图谱做辅助增强。内容小而美，逻辑关系清晰

2 列出mt_brand_data数据库中的表
问题：现在列出mt_brand_data数据库中的所有表，以确认RAG查询结果中的表确实存在。mt_brand_data包含具体的业务数据。根据知识库引导对数据的探索和利用
数据：mt_brand_data schema信息

3 注册相关的用户表
问题：想要注册 mt_brand_data: mt_members.csv
数据：mt_members.csv，并展示样例数据

4 查看表结构
问题：想要获取表结构 infinity_session: mt_brand_data_3ed5042183f4
数据：已注册的表